# Chatterbox (CPU-only) — fast debug loop

Use this notebook when you're blocked from a GPU runtime and just want to test 3-5 lines of text at a time. It's a separate file from your GPU notebook — that one is untouched.

Differences from the GPU notebook:
- Installs the CPU build of PyTorch (no CUDA), in its own micromamba env (`cb311-cpu`) so it never collides with your GPU env.
- **No Google Drive mount.** Generated audio is written to `/content/chatterbox_outputs` in the Colab VM instead. This means output does **not** persist across runtimes — download anything you want to keep before the session ends.
- No repo code changes. Your fork is cloned as-is; the server auto-detects there's no CUDA and runs on CPU.
- CPU thread count is tuned for the debug loop.
- Two optional/experimental cells at the bottom (`torch.compile`, dynamic quantization) are off by default — see the caveats in each before enabling.

## One-time setup

### 1. Environment + packages (micromamba, CPU-only PyTorch, Chatterbox)

In [ ]:
%%bash
set -euo pipefail

cd /content
MICROMAMBA="/content/bin/micromamba"
ENV_NAME="cb311-cpu"

ts() { date +"[%Y-%m-%d %H:%M:%S]"; }

# --- Create isolated Python 3.11 environment (skips if it already exists) ---
# Separate env name from the GPU notebook's "cb311", so switching between the two
# notebooks never means reinstalling torch.
if [ ! -x "$MICROMAMBA" ]; then
    echo "$(ts) Downloading micromamba..."
    curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
fi
if ! "$MICROMAMBA" env list | grep -q "$ENV_NAME"; then
    echo "$(ts) Creating $ENV_NAME environment..."
    "$MICROMAMBA" create -y -n "$ENV_NAME" -c conda-forge python=3.11 pip
else
    echo "$(ts) $ENV_NAME environment already exists, skipping creation."
fi

# --- Install PyTorch (CPU build) + Chatterbox (Turbo) ---
echo "$(ts) Upgrading pip tooling inside $ENV_NAME..."
"$MICROMAMBA" run -n "$ENV_NAME" python -m pip install -U pip setuptools wheel --progress-bar on

echo "$(ts) Installing PyTorch 2.5.1 (CPU build)... (this can take a while)"
"$MICROMAMBA" run -n "$ENV_NAME" pip install \
  --progress-bar on \
  torch==2.5.1 torchaudio==2.5.1 torchvision==0.20.1 \
  --index-url https://download.pytorch.org/whl/cpu

echo "$(ts) Installing Chatterbox package (from GitHub, no-cache, upgrade)..."
"$MICROMAMBA" run -n "$ENV_NAME" pip uninstall -y chatterbox-tts chatterbox || true
"$MICROMAMBA" run -n "$ENV_NAME" pip install \
  --no-cache-dir --upgrade \
  --progress-bar on \
  "chatterbox-tts @ git+https://github.com/devnen/chatterbox-v2.git@master"

echo "$(ts) Installing s3tokenizer + onnx (--no-deps to avoid protobuf conflict)..."
"$MICROMAMBA" run -n "$ENV_NAME" pip install --no-deps s3tokenizer==0.3.0 onnx==1.16.0

echo "$(ts) Force-upgrading protobuf for onnx compatibility..."
"$MICROMAMBA" run -n "$ENV_NAME" pip install --no-deps --force-reinstall "protobuf>=4.25.0"

echo "$(ts) ✅ Installation complete!"

# --- Verify install (no CUDA check here on purpose — this env has no CUDA wheels) ---
echo "$(ts) Verifying install..."
"$MICROMAMBA" run -n "$ENV_NAME" python - <<'PY'
import inspect, torch
import chatterbox.tts_turbo as t

print("✅ torch:", torch.__version__)
print("✅ cuda available:", torch.cuda.is_available(), "(expected False in this CPU env)")
print("✅ cpu count:", __import__('os').cpu_count())

src = inspect.getsource(t.ChatterboxTurboTTS.from_pretrained)

# Heuristic check for the common buggy pattern that forces token=True semantics
markers = [" or True", "token=True", "token = True", "use_auth_token=True"]
hits = [m for m in markers if m in src]
print("Heuristic auth-forcing markers found:", hits)

if hits:
    raise SystemExit(
        "\n❌ This install still appears to force HF auth. Re-run this cell.\n"
    )

print("✅ Looks good: Turbo should download without requiring user tokens.")
PY


### 2. Clone your fork, server deps, voice + config (local output, no Drive)

No `google.colab.drive` import here, no mount step. Generated audio is written to a local folder in the VM (`/content/chatterbox_outputs`) so there's nothing to authorize and no Drive dependency — just remember it disappears when the runtime recycles.

In [ ]:
# @title 2. One-time setup: clone repo, server deps, config (CPU, no Drive)
import os, shutil, subprocess, multiprocessing
from pathlib import Path
import yaml
from google.colab import userdata

# ==== EDIT THESE IF YOU WANT DIFFERENT DEFAULTS ====
PORT = 8005  # different from the GPU notebook's 8004, in case both are ever up at once
REPO_OWNER = "michtai"
REPO_NAME = "chatterbox"
REPO_DIR = Path(f"/content/{REPO_NAME}-cpu")  # separate checkout dir from the GPU notebook
ENV_NAME = "cb311-cpu"
LOCAL_OUTPUTS_DIR = Path("/content/chatterbox_outputs")  # NOT Google Drive — wiped when runtime recycles
VOICE_FILENAME = "delightful_really_soft.wav"            # committed under voices/ in your fork
CHUNK_SIZE = 100  # even smaller than the GPU notebook's 130 — you're testing short debug
                  # snippets on CPU, so there's no reason to risk multi-turn chunk boundaries
GENERATION_DEFAULTS = {
    "temperature": 0.65,
    "exaggeration": 0.4,
    "cfg_weight": 0.5,
    "seed": 1818,
    "speed_factor": 1.0,
}
CPU_THREADS = multiprocessing.cpu_count()  # match Colab's allocated vCPUs
# =====================================================

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")  # Colab secret, needed if the fork is private
CLONE_URL = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
LOG_STDOUT = "/content/chatterbox_server_stdout_cpu.log"

def sh(cmd, check=False, cwd=None):
    return subprocess.run(["bash", "-lc", cmd], check=check, cwd=cwd)

# === Local output folder (replaces Drive mount) ===
print("=== Creating local output folder (no Drive) ===")
LOCAL_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"✅ Generated audio will be saved to: {LOCAL_OUTPUTS_DIR}")
print("⚠️  This is local VM storage, not Drive — download anything you want to keep "
      "before the runtime disconnects or recycles.")

# === Clone your fork into a separate dir from the GPU notebook's checkout ===
print("\n=== Cloning your fork ===")
sh(f"rm -rf {REPO_DIR}", check=False)
sh(f"git clone {CLONE_URL} {REPO_DIR}", check=True)

print("\n=== Installing server requirements ===")
if (REPO_DIR / "requirements.txt").exists():
    sh(f"/content/bin/micromamba run -n {ENV_NAME} pip install -r requirements.txt", check=False, cwd=REPO_DIR)
else:
    sh(
        f"/content/bin/micromamba run -n {ENV_NAME} pip install "
        "fastapi 'uvicorn[standard]' pyyaml soundfile librosa safetensors "
        "python-multipart requests jinja2 watchdog aiofiles unidecode inflect tqdm "
        "pydub audiotsm praat-parselmouth",
        check=False, cwd=REPO_DIR
    )
sh(f"/content/bin/micromamba run -n {ENV_NAME} pip install --no-deps --force-reinstall 'protobuf>=4.25.0'", check=False)

# === Patch chatterbox to make Perth watermarker gracefully optional ===
# Same patch as the GPU notebook, applied to the installed package in THIS env
# (cb311-cpu). Not a repo change — it patches site-packages, same as before.
print("\n=== Applying watermarker patch ===")
SITE_PKG = f"/root/.local/share/mamba/envs/{ENV_NAME}/lib/python3.11/site-packages"
CB_DIR = Path(SITE_PKG) / "chatterbox"
SENTINEL = "# [patched: watermarker made optional]"
TARGET = "self.watermarker = perth.PerthImplicitWatermarker()"
patched = 0
for fname in ["tts.py", "tts_turbo.py", "mtl_tts.py", "vc.py"]:
    fp = CB_DIR / fname
    if not fp.exists():
        continue
    content = fp.read_text(encoding="utf-8")
    if SENTINEL in content or TARGET not in content:
        continue
    lines = content.split("\n")
    new_lines = []
    for line in lines:
        if TARGET in line and line.lstrip().startswith("self."):
            ind = line[:len(line) - len(line.lstrip())]
            new_lines.append(f"{ind}{SENTINEL}")
            new_lines.append(f"{ind}try:")
            new_lines.append(f"{ind}    self.watermarker = perth.PerthImplicitWatermarker()")
            new_lines.append(f"{ind}except Exception:")
            new_lines.append(f"{ind}    class _NoOpWatermarker:")
            new_lines.append(f"{ind}        def apply_watermark(self, wav, *args, **kwargs):")
            new_lines.append(f"{ind}            return wav")
            new_lines.append(f"{ind}    self.watermarker = _NoOpWatermarker()")
        else:
            new_lines.append(line)
    fp.write_text("\n".join(new_lines), encoding="utf-8")
    print(f"  Patched {fname}")
    patched += 1
print(f"  {patched} file(s) patched for optional watermarking" if patched else "  No patching needed")

# === Pin CPU thread count for this env ===
# Written into a sitecustomize.py so it applies automatically to every python
# process launched in cb311-cpu (including server.py), with no repo edits.
print("\n=== Configuring CPU thread count ===")
sitecustomize_path = Path(SITE_PKG) / "sitecustomize.py"
sitecustomize_path.write_text(
    "import os\n"
    f"os.environ.setdefault('OMP_NUM_THREADS', '{CPU_THREADS}')\n"
    f"os.environ.setdefault('MKL_NUM_THREADS', '{CPU_THREADS}')\n"
    "try:\n"
    "    import torch\n"
    f"    torch.set_num_threads({CPU_THREADS})\n"
    "except Exception:\n"
    "    pass\n",
    encoding="utf-8",
)
print(f"  ✅ Pinned to {CPU_THREADS} threads (matches this Colab instance's vCPUs)")


def apply_voice_and_config():
    """
    Copies the committed voice into reference_audio/ (if needed) and writes
    config.yaml with the active model, voice, local output dir, and generation
    defaults. Called here for initial setup, and again by the "run server" cell
    after every `git pull`, since config.yaml is a tracked file and a pull could
    otherwise overwrite these local settings.
    """
    voices_dir = REPO_DIR / "voices"
    reference_dir = REPO_DIR / "reference_audio"
    reference_dir.mkdir(parents=True, exist_ok=True)

    voice_in_voices_dir = voices_dir / VOICE_FILENAME
    if not voice_in_voices_dir.exists():
        raise SystemExit(
            f"Expected {voice_in_voices_dir} in the cloned repo but it's missing. "
            f"Check that {VOICE_FILENAME} is actually committed under voices/ in your fork."
        )
    if not (reference_dir / VOICE_FILENAME).exists():
        shutil.copy(voice_in_voices_dir, reference_dir / VOICE_FILENAME)

    config_path = REPO_DIR / "config.yaml"
    with open(config_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    cfg.setdefault("model", {})
    cfg["model"]["repo_id"] = "chatterbox"  # Default active model: Chatterbox Original (English)

    cfg.setdefault("tts_engine", {})
    cfg["tts_engine"]["default_voice_id"] = VOICE_FILENAME

    cfg.setdefault("server", {})
    cfg["server"]["port"] = PORT  # keep config.yaml in sync with this notebook's PORT

    cfg.setdefault("paths", {})
    cfg["paths"]["output"] = str(LOCAL_OUTPUTS_DIR)  # local VM path, not Drive

    cfg.setdefault("audio_output", {})
    # False on purpose: the "generate_tts()" cell below saves the file itself
    # (with a nicer name built from the text + elapsed time + timestamp). If the
    # server ALSO saves to disk here, you end up with two copies of every clip.
    cfg["audio_output"]["save_to_disk"] = False

    cfg.setdefault("generation_defaults", {})
    cfg["generation_defaults"].update(GENERATION_DEFAULTS)

    cfg.setdefault("ui_state", {})
    cfg["ui_state"]["last_text"] = "Type your text here."
    cfg["ui_state"]["last_voice_mode"] = "predefined"
    cfg["ui_state"]["last_predefined_voice"] = VOICE_FILENAME
    cfg["ui_state"]["last_reference_file"] = VOICE_FILENAME
    cfg["ui_state"]["last_seed"] = GENERATION_DEFAULTS["seed"]
    cfg["ui_state"]["last_chunk_size"] = CHUNK_SIZE
    cfg["ui_state"]["last_split_text_enabled"] = True

    with open(config_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, default_flow_style=False, sort_keys=False)

    print(f"  ✅ Active model: {cfg['model']['repo_id']} (Chatterbox Original / English)")
    print(f"  ✅ Output directory: {LOCAL_OUTPUTS_DIR}")
    print(f"  ✅ Default voice: {VOICE_FILENAME}")
    print(f"  ✅ Server port: {PORT}")
    print(f"  ✅ Chunk size: {CHUNK_SIZE}")
    print(f"  ✅ Generation defaults: {GENERATION_DEFAULTS}")
    print(f"  ✅ CPU threads: {CPU_THREADS}")


print("\n=== Configuring voice + output folder + generation defaults ===")
apply_voice_and_config()

print("\n✅ One-time setup complete. Run the next cell to start the server.")


## (Optional / experimental) Extra CPU tricks

**Off by default — read the caveats before turning either on.** These are more aggressive than thread tuning and can just as easily make your specific debug loop slower. Try them only if the plain CPU path above still feels too slow.

- `ENABLE_TORCH_COMPILE`: wraps the loaded model in `torch.compile()`. This model has variable-length autoregressive sampling, so it tends to **recompile on shape changes**, and a 3-5 line debug loop rarely repeats the exact same shape twice — you may pay compile overhead on every run instead of saving time.
- `ENABLE_DYNAMIC_QUANTIZATION`: converts `nn.Linear` weights to int8 after the model loads. Wrapped in try/except so a failure just falls back to the unquantized model, but quantization support for this architecture isn't guaranteed, and audio quality on a quantized model hasn't been validated here — treat any output from this mode as unverified.

Both apply via the same `sitecustomize.py` mechanism as thread pinning (no repo edits), by monkeypatching `ChatterboxTTS.from_pretrained` / `ChatterboxTurboTTS.from_pretrained` after they build the model.

In [ ]:
# @title (Optional) Enable torch.compile and/or dynamic quantization — leave both False unless you've read the caveats above
ENABLE_TORCH_COMPILE = False
ENABLE_DYNAMIC_QUANTIZATION = False

from pathlib import Path
import textwrap

SITE_PKG = f"/root/.local/share/mamba/envs/{ENV_NAME}/lib/python3.11/site-packages"
sitecustomize_path = Path(SITE_PKG) / "sitecustomize.py"
base = sitecustomize_path.read_text(encoding="utf-8") if sitecustomize_path.exists() else ""

if ENABLE_TORCH_COMPILE or ENABLE_DYNAMIC_QUANTIZATION:
    # The two booleans are baked in as literals below, so the generated
    # sitecustomize.py has no dependency on this notebook's variables.
    patch_src = textwrap.dedent(f"""
        # --- optional experimental patches ---
        def _patch_chatterbox_experimental():
            import torch
            try:
                import chatterbox.tts_turbo as _t
            except Exception:
                return
            _orig = _t.ChatterboxTurboTTS.from_pretrained
            def _wrapped(*args, **kwargs):
                model = _orig(*args, **kwargs)
                if {ENABLE_DYNAMIC_QUANTIZATION}:
                    try:
                        for attr_name in dir(model):
                            sub = getattr(model, attr_name, None)
                            if isinstance(sub, torch.nn.Module):
                                quantized = torch.quantization.quantize_dynamic(
                                    sub, {{torch.nn.Linear}}, dtype=torch.qint8
                                )
                                setattr(model, attr_name, quantized)
                        print('[experimental] dynamic quantization applied')
                    except Exception as e:
                        print('[experimental] quantization failed, continuing unquantized:', repr(e))
                if {ENABLE_TORCH_COMPILE}:
                    try:
                        for attr_name in dir(model):
                            sub = getattr(model, attr_name, None)
                            if isinstance(sub, torch.nn.Module):
                                setattr(model, attr_name, torch.compile(sub))
                        print('[experimental] torch.compile applied')
                    except Exception as e:
                        print('[experimental] torch.compile failed, continuing uncompiled:', repr(e))
                return model
            _t.ChatterboxTurboTTS.from_pretrained = staticmethod(_wrapped)
        _patch_chatterbox_experimental()
    """)
    sitecustomize_path.write_text(base + patch_src, encoding="utf-8")
    print("✅ Experimental patch(es) written. Restart the server cell below to pick them up.")
else:
    print("Both flags are False — nothing changed. Set one to True above and re-run this cell to enable it.")


## Run the server

Re-run this cell any time: to start the server, restart it, or pick up new commits pushed to your fork (it runs `git fetch` + `git reset --hard origin/HEAD` before launching, then re-applies your voice/config settings since `config.yaml` is tracked).

**Tunnel choice** -- set `TUNNEL_PROVIDER` inside the cell to one of:
- `"none"` -- no browser access, use cell 4's direct API call only.
- `"localtunnel"` *(default)* -- no account needed, no 100-second limit, works the same in every browser (including Firefox). Occasionally flaky, and shows a one-time interstitial page that may ask for a "tunnel password" -- the cell prints how to get it when this happens.
- `"cloudflare"` -- very reliable, no account needed, but hard-caps any single request at ~100 seconds (`HTTP 524`). Fine for browsing the UI, risky for triggering a generation from the browser on CPU.

**This cell now finishes on its own** once the server reports itself ready (it launches the server as a detached background process rather than streaming its logs forever), so you can go straight to cell 4 -- or the stop-server cell -- right after, no need to interrupt anything. Live logs are written to the log file the whole time; tail it with `!tail -f /content/chatterbox_server_stdout_cpu.log` in a scratch cell if you want to watch progress.

In [ ]:
# @title 3. Run server (CPU) — pulls latest changes, starts the server in the
# background, and returns control as soon as it's ready (so you can run
# cell 4 or any other cell right after, without interrupting anything).
import os, re, socket, subprocess, threading, time
import requests
from pathlib import Path

# "none"        -- no tunnel at all. Use cell 4 (direct 127.0.0.1 API call) only.
# "localtunnel" -- no account needed, no 100s timeout, but occasionally flaky and
#                  shows a one-time "click to continue" interstitial in the browser
#                  (may ask for a "tunnel password" -- printed below when needed).
# "cloudflare"  -- no account needed, very reliable, but hard-caps any single
#                  request at ~100 seconds (HTTP 524) -- avoid for slow generations.
TUNNEL_PROVIDER = "localtunnel"

# How long to wait for the model to finish loading before giving up.
MAX_WAIT_SECONDS = 600

CLOUDFLARED = "/content/bin/cloudflared"
CLOUDFLARE_URL_RE = re.compile(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com")
LOCALTUNNEL_URL_RE = re.compile(r"https://[a-zA-Z0-9\-]+\.loca\.lt")

def port_open(host="127.0.0.1", port=PORT, timeout=0.25):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

def ensure_cloudflared():
    """Download the cloudflared binary once per runtime (free, no account needed)."""
    if os.path.exists(CLOUDFLARED):
        return
    print("=== Downloading cloudflared (one-time, free, no account needed) ===")
    Path("/content/bin").mkdir(parents=True, exist_ok=True)
    sh(
        "curl -Ls https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
        f"-o {CLOUDFLARED} && chmod +x {CLOUDFLARED}",
        check=True,
    )
    print("✅ cloudflared installed")

def ensure_node():
    """Make sure node/npx exist (needed to run localtunnel via `npx localtunnel`)."""
    result = subprocess.run(["bash", "-lc", "command -v npx"], capture_output=True, text=True)
    if result.returncode == 0:
        return
    print("=== Installing Node.js (one-time, needed for localtunnel) ===")
    sh("apt-get update -qq && apt-get install -y -qq nodejs npm", check=True)
    print("✅ Node.js installed")

print("=== Pulling latest changes from your fork ===")
sh("git fetch origin", check=True, cwd=REPO_DIR)
sh("git reset --hard origin/HEAD", check=True, cwd=REPO_DIR)
sh("git log -1 --oneline", check=False, cwd=REPO_DIR)

print("\n=== Re-applying voice + config (in case the pull touched config.yaml) ===")
apply_voice_and_config()

print("\n=== Removing old stdout log ===")
Path(LOG_STDOUT).unlink(missing_ok=True)

# Kill any leftover server/tunnel from a previous run of this cell before starting fresh.
sh(f"lsof -t -i:{PORT} | xargs -r kill -9", check=False)
sh("pkill -f 'cloudflared tunnel' || true", check=False)
sh("pkill -f 'localtunnel' || true", check=False)

if TUNNEL_PROVIDER == "cloudflare":
    ensure_cloudflared()
elif TUNNEL_PROVIDER == "localtunnel":
    ensure_node()
else:
    print("\n=== No tunnel selected (TUNNEL_PROVIDER = 'none') ===")
    print("Use cell 4 (local API call) for generation -- it talks to 127.0.0.1 directly.")

print("\n=== Starting server (CPU) in the background ===")
print("Log file:", LOG_STDOUT)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["OMP_NUM_THREADS"] = str(CPU_THREADS)
env["MKL_NUM_THREADS"] = str(CPU_THREADS)

# Put HF cache somewhere inspectable/persistent for this runtime
env["HF_HOME"] = "/content/hf_home_cpu"
env["TRANSFORMERS_CACHE"] = "/content/hf_home_cpu/transformers"
env["HF_HUB_CACHE"] = "/content/hf_home_cpu/hub"
Path(env["HF_HOME"]).mkdir(parents=True, exist_ok=True)

# Launch the server fully detached: its stdout/stderr are wired straight to
# the log file at the OS level (not read line-by-line by this cell), and
# start_new_session=True puts it in its own process group. That means this
# cell doesn't have to stay alive babysitting the process -- it just checks
# in on it periodically below, and finishes on its own once the server is up.
log_f = open(LOG_STDOUT, "w", encoding="utf-8", errors="replace")
proc = subprocess.Popen(
    ["/content/bin/micromamba", "run", "-n", ENV_NAME, "python", "-u", "server.py"],
    stdout=log_f,
    stderr=subprocess.STDOUT,
    env=env,
    cwd=REPO_DIR,
    start_new_session=True,
)
log_f.close()  # the child keeps its own duplicated handle; we don't need ours
print(f"Server process started (pid {proc.pid}).")

tunnel_proc = None
_tunnel_state = {"url": None}

if TUNNEL_PROVIDER == "cloudflare":
    tunnel_proc = subprocess.Popen(
        [CLOUDFLARED, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        start_new_session=True,
    )

    def _watch_tunnel():
        for line in tunnel_proc.stdout:
            m = CLOUDFLARE_URL_RE.search(line)
            if m and not _tunnel_state["url"]:
                _tunnel_state["url"] = m.group(0)
                break

    threading.Thread(target=_watch_tunnel, daemon=True).start()

elif TUNNEL_PROVIDER == "localtunnel":
    tunnel_proc = subprocess.Popen(
        ["npx", "--yes", "localtunnel", "--port", str(PORT)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        start_new_session=True,
    )

    def _watch_tunnel():
        for line in tunnel_proc.stdout:
            m = LOCALTUNNEL_URL_RE.search(line)
            if m and not _tunnel_state["url"]:
                _tunnel_state["url"] = m.group(0)
                break

    threading.Thread(target=_watch_tunnel, daemon=True).start()

# === Wait (bounded) for the server to come up, then hand control back ===
print(f"Waiting for the model to load (up to {MAX_WAIT_SECONDS}s on CPU) ", end="", flush=True)
wait_start = time.time()
ready = False
while time.time() - wait_start < MAX_WAIT_SECONDS:
    if port_open():
        ready = True
        break
    if proc.poll() is not None:
        break
    print(".", end="", flush=True)
    time.sleep(3)

print()  # newline after the dots

if not ready:
    if proc.poll() is not None:
        print(f"\n❌ Server process exited early with code {proc.returncode}.")
    else:
        print(f"\n⚠️  Still not up after {MAX_WAIT_SECONDS}s.")
    print(f"   Check the log for details: {LOG_STDOUT}")
    print(f"   (e.g. run: !tail -n 60 {LOG_STDOUT})")
else:
    print("="*60)
    print("=== Server is ready! (CPU mode) ===")
    print("="*60)

    if TUNNEL_PROVIDER in ("cloudflare", "localtunnel"):
        wait_start2 = time.time()
        while _tunnel_state["url"] is None and time.time() - wait_start2 < 20:
            if tunnel_proc.poll() is not None:
                break
            time.sleep(0.25)

        if _tunnel_state["url"]:
            public_url = _tunnel_state["url"]
            print(f"\n🌐 Open this URL in any browser:\n\n   {public_url}\n")
            if TUNNEL_PROVIDER == "cloudflare":
                print(f"📚 API docs:  {public_url}/docs\n")
                print("⚠️  Reminder: any single request over ~100s will fail with HTTP 524 --\n"
                      "   use cell 4 for actual generation, not the browser, if text is long.\n")
            else:  # localtunnel
                print("📚 API docs:  " + public_url + "/docs\n")
                print("⚠️  First time opening this URL, localtunnel may show an interstitial\n"
                      "   page asking for a 'Tunnel Password'. If so, that password is just\n"
                      "   this VM's public IP -- fetch it by running in a new cell:\n\n"
                      "     !curl -s https://loca.lt/mytunnelpassword\n\n"
                      "   Paste the result into the browser prompt, and you're through --\n"
                      "   no 100-second limit here, so long CPU generations are fine.\n")
        else:
            print(
                f"\n⚠️  {TUNNEL_PROVIDER} tunnel URL not detected within 20s.\n"
                "   It may still be starting — check the log for a tunnel URL, or re-run this cell.\n"
            )
    else:
        print(f"\n🖥️  No tunnel started. Server is local-only at http://127.0.0.1:{PORT}\n")

    try:
        mi = requests.get(f"http://127.0.0.1:{PORT}/api/model-info", timeout=2).json()
        print("/api/model-info:", mi)
    except Exception as e:
        print("/api/model-info query failed:", repr(e))

    print("="*60)
    print(f"\n✅ This cell has finished -- the server keeps running in the background.")
    print(f"   You can now run cell 4 (direct API call) or any other cell immediately.")
    print(f"   Full/live logs any time: !tail -f {LOG_STDOUT}")


## Generate audio directly (bypasses the Cloudflare tunnel)

Use this cell instead of the web UI for your debug loop. It calls the local
server straight on `127.0.0.1` from inside this notebook/VM, so the request
never goes through the `trycloudflare.com` tunnel at all.

Why this matters: Cloudflare's free quick tunnels kill any request that
takes longer than 100 seconds (`HTTP 524`), and Chatterbox on 2 CPU threads
routinely takes longer than that. Talking to the server directly has no such
limit — it'll just take as long as it takes, and you'll still get your audio
back and hear it inline below.

The tunnel URL is still useful for browsing the web UI itself; just don't
click "Generate" in the browser for anything that might run long.


In [ ]:
# @title 4. Generate speech via local API call (no tunnel, no 100s limit)
import time
import re
import requests
from pathlib import Path
from IPython.display import Audio, display, Javascript

#####################################################################
#####################################################################
#####################################################################
TTS_TEXT = """
"You were told to bear only daughters to the Atraydeze."
"""  # <-- edit this each debug run
#####################################################################
#####################################################################
#####################################################################


def _notify_done():
    """
    Plays a short beep in the browser tab when generation finishes.
    Uses a tiny synthesized tone (no internet/network calls, no permissions
    needed) via the Web Audio API, so it works even if the tab isn't focused.
    """
    display(Javascript("""
        (function() {
            try {
                const ctx = new (window.AudioContext || window.webkitAudioContext)();
                const osc = ctx.createOscillator();
                const gain = ctx.createGain();
                osc.connect(gain);
                gain.connect(ctx.destination);
                osc.frequency.value = 880;
                gain.gain.setValueAtTime(0.15, ctx.currentTime);
                osc.start();
                osc.stop(ctx.currentTime + 0.18);
            } catch (e) {}
        })();
    """))


def generate_tts(text=None, save=True, play=True, timeout=600):
    """
    Calls the local TTS server directly on 127.0.0.1, bypassing the
    cloudflared tunnel entirely. Since this request never touches Cloudflare,
    it isn't subject to the free quick tunnel's 100-second gateway timeout --
    only the local `timeout` below (default 10 min) applies.
    """
    text = text if text is not None else TTS_TEXT

    payload = {
        "text": text,
        "voice_mode": "predefined",
        "predefined_voice_id": VOICE_FILENAME,
        "output_format": "wav",
        "split_text": True,
        "chunk_size": CHUNK_SIZE,
        "temperature": GENERATION_DEFAULTS["temperature"],
        "exaggeration": GENERATION_DEFAULTS["exaggeration"],
        "cfg_weight": GENERATION_DEFAULTS["cfg_weight"],
        "seed": GENERATION_DEFAULTS["seed"],
        "speed_factor": GENERATION_DEFAULTS["speed_factor"],
    }

    url = f"http://127.0.0.1:{PORT}/tts"
    print(f"Requesting TTS for {len(text)} chars from {url} (local, no tunnel) ...")
    start = time.time()
    try:
        resp = requests.post(url, json=payload, timeout=timeout)
    except requests.exceptions.RequestException as e:
        print(f"❌ Request failed after {time.time() - start:.1f}s: {e!r}")
        print("   (Is the server cell above still running?)")
        return None
    elapsed = time.time() - start

    if resp.status_code != 200:
        print(f"❌ Server returned HTTP {resp.status_code} after {elapsed:.1f}s")
        print(resp.text[:1000])
        return None

    print(f"✅ Got {len(resp.content)} bytes of audio in {elapsed:.1f}s")

    out_path = None
    if save:
        LOCAL_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
        # This is now the ONLY place a file gets written -- server-side
        # save_to_disk is off (see cell 2), so there's no duplicate copy.
        # Build filename from the first 3 words of the text being spoken.
        words = re.findall(r"[A-Za-z0-9']+", text)[:3]
        prefix = "_".join(words) if words else "debug"
        out_path = LOCAL_OUTPUTS_DIR / f"{prefix}_{elapsed:.1f}s_{int(time.time())}.wav"
        out_path.write_bytes(resp.content)
        print(f"💾 Saved to {out_path}")

    if play:
        display(Audio(resp.content))

    _notify_done()  # audible beep so you know it's done without watching the tab

    return out_path


# Run a generation with the text set in TTS_TEXT above.
# Re-run this cell (edit TTS_TEXT first) for each new debug snippet --
# no need to touch the browser UI or the tunnel at all.
generate_tts()


## Stop the server

Run any time you want to free the port — e.g. before re-running the cell above.

In [ ]:
%%bash
PORT=8005

echo "PIDs listening on port $PORT:"
sudo lsof -t -i:$PORT || true

echo "Killing server..."
sudo lsof -t -i:$PORT | xargs -r sudo kill -9

echo "Killing cloudflared tunnel..."
pkill -f "cloudflared tunnel" || true

echo "Killing localtunnel..."
pkill -f "localtunnel" || true

echo "Verify nothing is listening:"
sudo lsof -i:$PORT || true
